# Equitable mentorship pairing engine — data analysis

Analysis for the Catalyst Strategists team project (UN SDG Goal 5, gender equality). This notebook examines real mentor-mentee relationship data alongside current developer survey data to quantify the gender gap in technical mentorship access.

**Data sources:** [Academic mentorship dataset](https://doi.org/10.5281/zenodo.4917086) (743k+ real pairs, Nature Scientific Data) and the [Stack Overflow 2025 Developer Survey](https://www.kaggle.com/datasets/edoardogalli/stack-overflow-annual-developer-survey-2025).

## Checking file paths

In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/first_name_gender.csv
/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/researcher.csv
/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/gender_pairing_summary.csv
/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/mentorship.csv
/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/mentor_gender_totals.csv
/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/stack_overflow/survey_results_schema.csv
/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/stack_overflow/survey_results_public.csv
/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine/stack_overflow/2025 Developer Survey Tool.pdf


## Loading the data

In [4]:
import pandas as pd

base = '/kaggle/input/datasets/israelbabalola0/equitable-mentorship-pairing-engine'

researcher = pd.read_csv(f'{base}/researcher.csv')
mentorship = pd.read_csv(f'{base}/mentorship.csv')
gender = pd.read_csv(f'{base}/first_name_gender.csv')

so_survey = pd.read_csv(f'{base}/stack_overflow/survey_results_public.csv')
so_schema = pd.read_csv(f'{base}/stack_overflow/survey_results_schema.csv')

print("All loaded:", researcher.shape, mentorship.shape, gender.shape, so_survey.shape, so_schema.shape)

All loaded: (774733, 9) (743176, 8) (91062, 2) (49123, 170) (139, 6)


## Merging mentee and mentor names and gender

In [5]:
mentee_info = researcher[['PID', 'FirstName']].rename(
    columns={'PID': 'MenteeID', 'FirstName': 'MenteeFirstName'}
)
merged = mentorship.merge(mentee_info, on='MenteeID', how='left')

mentor_info = researcher[['PID', 'FirstName']].rename(
    columns={'PID': 'MentorID', 'FirstName': 'MentorFirstName'}
)
merged = merged.merge(mentor_info, on='MentorID', how='left')

mentee_gender = gender.rename(columns={'FirstName': 'MenteeFirstName', 'gender': 'MenteeGender'})
merged = merged.merge(mentee_gender, on='MenteeFirstName', how='left')

mentor_gender = gender.rename(columns={'FirstName': 'MentorFirstName', 'gender': 'MentorGender'})
merged = merged.merge(mentor_gender, on='MentorFirstName', how='left')

merged[['MenteeFirstName', 'MenteeGender', 'MentorFirstName', 'MentorGender']].head(10)

,MenteeFirstName,MenteeGender,MentorFirstName,MentorGender
0,BENJAMIN,man,JACK,man
1,BENJAMIN,man,JACK,man
2,RYAN,man,JACK,man
3,ALAN,man,MELVIN,man
4,C,unknown,DAVID,man
5,JACK,man,DAVID,man
6,JACK,man,WENDELL,man
7,ANITHA,woman,EARL,man
8,EARL,man,CHARLES,man
9,EARL,man,ROBERT,man


## Gender representation among mentors

In [6]:
# Gender pairing breakdown
pairing_counts = merged.groupby(['MenteeGender', 'MentorGender']).size().reset_index(name='PairCount')
pairing_counts['Percent'] = (pairing_counts['PairCount'] / pairing_counts['PairCount'].sum() * 100).round(2)
pairing_counts = pairing_counts.sort_values('PairCount', ascending=False)

# Mentor gender totals
mentor_totals = merged['MentorGender'].value_counts(normalize=True).mul(100).round(2).reset_index()
mentor_totals.columns = ['MentorGender', 'PercentOfAllMentors']

print(pairing_counts)
print(mentor_totals)

  MenteeGender MentorGender  PairCount  Percent
0          man          man     274479    36.95
6        woman          man     146238    19.69
8        woman        woman      85729    11.54
3      unknown          man      83007    11.17
2          man        woman      50853     6.85
1          man      unknown      34254     4.61
4      unknown      unknown      26673     3.59
5      unknown        woman      20986     2.83
7        woman      unknown      20608     2.77
  MentorGender  PercentOfAllMentors
0          man                67.81
1        woman                21.21
2      unknown                10.98


## Does mentorship access matter more to early-career developers?

In [7]:
so_survey['WorkExp'] = pd.to_numeric(so_survey['WorkExp'], errors='coerce')
so_survey['CareerStage'] = so_survey['WorkExp'].apply(
    lambda x: 'Early-career (0-3 yrs)' if pd.notna(x) and x <= 3 else ('Experienced (4+ yrs)' if pd.notna(x) else 'Unknown')
)

jobsat_cols = [c for c in so_survey.columns if c.startswith('JobSatPoints_') and not c.endswith('_TEXT')]
labels = so_schema[so_schema['question'].str.contains('Rank the following attributes', na=False)][['qname', 'sub']]
labels = labels.rename(columns={'qname': 'Attribute'}).drop_duplicates(subset='Attribute')

comparison = so_survey.groupby('CareerStage')[['JobSatPoints_4', 'JobSatPoints_5']].mean().round(2)
comparison.columns = ['ExpertMentors_AvgRank', 'OpportunityToMentor_AvgRank']
comparison = comparison.reset_index()
comparison

,CareerStage,ExpertMentors_AvgRank,OpportunityToMentor_AvgRank
0,Early-career (0-3 yrs),8.04,10.26
1,Experienced (4+ yrs),9.84,9.68
2,Unknown,7.30,9.66


## Findings

- **Mentors are overwhelmingly men.** 67.81% of mentors in this dataset are men, compared to 21.21% women (10.98% of names couldn't be confidently classified by the gender-inference model).
- **The pairing pattern skews further.** Man-to-man mentorship makes up 36.95% of all pairs; woman-to-woman is only 11.54%.
- **Early-career developers value mentorship access more.** On a 13-point importance ranking, "expert mentors" scores 8.04 for developers with 0–3 years of experience, versus 9.84 for those with 4+ years (lower = ranked more important).
- **The 2025 Stack Overflow survey no longer asks about gender**, so it's used here only for career-stage context, not representation figures.

## Recommendations

- Weight topic or skill similarity between mentor and mentee, since real mentorship data shows this pattern already forms naturally in relationships that succeed.
- Don't rely on informal, organic introductions to fill mentor roles for women — the data shows that process already produces a skewed outcome.
- Target mentor recruiting specifically at senior women, since they're the scarce side of this pairing, not the mentees.
- Make demographic fields in the matching engine self-reported and optional, rather than inferred, given the documented misclassification issues in this dataset.

## Limitations

Gender in the mentorship dataset is inferred from first names by a model, not self-reported. The mentorship data itself comes from bioscience and neuroscience academia, not the tech industry, so it's a structural stand-in for how mentorship relationships form rather than a tech-specific sample.

## Is the gender-pairing pattern statistically significant?

In [8]:
import pandas as pd
import scipy.stats as stats

# 1. Create the contingency table (Observed frequencies)
contingency_table = pd.crosstab(merged['MenteeGender'], merged['MentorGender'])
print("--- Observed Contingency Table ---")
print(contingency_table)
print("\n" + "="*40 + "\n")

# 2. Run the Chi-Square Test of Independence
chi2, p, dof, expected_frequencies = stats.chi2_contingency(contingency_table)
expected_df = pd.DataFrame(
    expected_frequencies,
    index=contingency_table.index,
    columns=contingency_table.columns)
print("--- Expected Contingency Table (If Purely Random) ---")
print(expected_df.round(2))
print("\n" + "="*40 + "\n")

# 3. Print Modeling Metrics
print("--- Chi-Square Test Results ---")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"Degrees of Freedom:  {dof}")
print(f"P-value:             {p:.4e}")

# 4. Interpret Significance
alpha = 0.05
if p < alpha:
    print("\nResult: STATISTICALLY SIGNIFICANT.")
    print(f"Reject the null hypothesis at alpha={alpha}. Gender pairing is NOT random.")
else:
    print("\nResult: NOT STATISTICALLY SIGNIFICANT.")
    print(f"Fail to reject the null hypothesis at alpha={alpha}. Pairings match random baseline expectations.")

--- Observed Contingency Table ---
MentorGender     man  unknown  woman
MenteeGender                        
man           274479    34254  50853
unknown        83007    26673  20986
woman         146238    20608  85729


--- Expected Contingency Table (If Purely Random) ---
MentorGender        man   unknown     woman
MenteeGender                               
man           243841.57  39469.28  76275.16
unknown        88606.90  14342.31  27716.79
woman         171275.53  27723.42  53576.05


--- Chi-Square Test Results ---
Chi-Square Statistic: 50383.7359
Degrees of Freedom:  4
P-value:             0.0000e+00

Result: STATISTICALLY SIGNIFICANT.
Reject the null hypothesis at alpha=0.05. Gender pairing is NOT random.
